# Publication figures — acoustic anisotropy

Generates and customizes the manuscript figures for *Engineering Fracture Mechanics*.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ardominguezm/acoustic-anisotropy/blob/main/notebooks/03_publication_figures.ipynb)

**Inputs required:** `Supporting data for Figure 9.xlsx` and `Supporting data for Figure 13.xlsx` from Zenodo record 18501172. Raw source data are intentionally not redistributed in this repository.

Edit only the **CONFIG** block below to change fonts, sizes, line widths, marker sizes, figure dimensions, output formats, and file names.

In [ ]:
from pathlib import Path
import os, io, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# =========================
# CONFIG — edit this block
# =========================
CONFIG = {
    'data_mode': 'upload',       # 'upload' or 'drive'
    'drive_data_dir': '/content/drive/MyDrive/acoustic-anisotropy/data',
    'output_dir': '/content/figures',
    'font_family': 'DejaVu Sans',
    'font_size': 11,
    'title_size': 12,
    'label_size': 11,
    'tick_size': 10,
    'legend_size': 9,
    'line_width': 1.8,
    'marker_size': 6,
    'capsize': 4,
    'dpi': 500,
    'formats': ['pdf', 'png'],
    'fig1_size': (11.0, 4.5),
    'fig2_size': (7.0, 4.5),
    'fig3_size': (7.0, 4.5),
    'fig4_size': (6.5, 4.5),
    'show_titles': False,       # journal-style default
    'transparent': False,
}

plt.rcParams.update({
    'font.family': CONFIG['font_family'],
    'font.size': CONFIG['font_size'],
    'axes.titlesize': CONFIG['title_size'],
    'axes.labelsize': CONFIG['label_size'],
    'xtick.labelsize': CONFIG['tick_size'],
    'ytick.labelsize': CONFIG['tick_size'],
    'legend.fontsize': CONFIG['legend_size'],
    'axes.linewidth': 0.8,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

OUT = Path(CONFIG['output_dir'])
OUT.mkdir(parents=True, exist_ok=True)
print('Output directory:', OUT)


## 1. Load the two source-data files

With `data_mode='upload'`, Colab will ask you to upload the two Excel files. With `data_mode='drive'`, the notebook mounts Google Drive and looks in `drive_data_dir`.

In [ ]:
REQ = ['Supporting data for Figure 9.xlsx', 'Supporting data for Figure 13.xlsx']

if CONFIG['data_mode'] == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path(CONFIG['drive_data_dir'])
elif CONFIG['data_mode'] == 'upload':
    from google.colab import files
    print('Upload these files:')
    for x in REQ: print(' -', x)
    uploaded = files.upload()
    DATA_DIR = Path('/content')
else:
    raise ValueError("CONFIG['data_mode'] must be 'upload' or 'drive'")

paths = {name: DATA_DIR / name for name in REQ}
missing = [str(p) for p in paths.values() if not p.exists()]
assert not missing, f'Missing files: {missing}'
F9 = paths[REQ[0]]
F13 = paths[REQ[1]]
print('Figure 9 source:', F9)
print('Figure 13 source:', F13)


## 2. Reconstruct mechanical and AE trajectories

Failure time is defined as the time of peak load. AE-energy observations after peak load are excluded from the pre-failure figures.

In [ ]:
ANGLES = [0, 30, 45, 60, 90]
data = {}

for angle in ANGLES:
    df = pd.read_excel(F9, sheet_name=str(angle), header=None)
    load_t = pd.to_numeric(df.iloc[2:, 2], errors='coerce').to_numpy(float)
    load   = pd.to_numeric(df.iloc[2:, 3], errors='coerce').to_numpy(float)
    m = np.isfinite(load_t) & np.isfinite(load)
    load_t, load = load_t[m], load[m]
    i_peak = int(np.argmax(load))
    tf = float(load_t[i_peak])
    peak = float(load[i_peak])

    ae_t = pd.to_numeric(df.iloc[2:, 4], errors='coerce').to_numpy(float)
    ae_e = pd.to_numeric(df.iloc[2:, 5], errors='coerce').to_numpy(float)
    m = np.isfinite(ae_t) & np.isfinite(ae_e) & (ae_t >= 0) & (ae_e >= 0) & (ae_t <= tf)
    ae_t, ae_e = ae_t[m], ae_e[m]

    data[angle] = dict(load_t=load_t, load=load, tf=tf, peak=peak, ae_t=ae_t, ae_e=ae_e)

audit = pd.DataFrame([
    {'angle': a, 'failure_time_s': data[a]['tf'], 'peak_load': data[a]['peak'], 'n_ae_events': len(data[a]['ae_t'])}
    for a in ANGLES
])
audit


In [ ]:
mode_rows = []
for angle in ANGLES:
    df = pd.read_excel(F13, sheet_name=str(angle), header=None)
    n_tensile = pd.to_numeric(df.iloc[3:, 2], errors='coerce').notna().sum()
    n_shear   = pd.to_numeric(df.iloc[3:, 4], errors='coerce').notna().sum()
    mode_rows.append({
        'angle': angle,
        'n_tensile': int(n_tensile),
        'n_shear': int(n_shear),
        'tensile_fraction': n_tensile / (n_tensile + n_shear),
    })
mode = pd.DataFrame(mode_rows)
mode


## 3. Validated derived summaries used in Figs. 2–4

These values come from the frozen confirmatory/mechanistic analysis in this repository. They are kept explicit here so figure customization cannot silently change the scientific results.

In [ ]:
peak_stage = pd.DataFrame({
    'angle': [0, 30, 45, 60, 90],
    'stage_of_peak_energy_rate': [4, 4, 4, 3, 3],
})

bootstrap = pd.DataFrame([
    [0,  2.308424, -1.147626, 5.771095],
    [30, 1.137907, -2.271852, 4.923506],
    [45, 2.895149,  0.686460, 5.110506],
    [60, 3.979712,  2.309461, 5.759729],
    [90, 3.358227,  2.126741, 4.988488],
], columns=['angle', 'p_hat', 'ci_low', 'ci_high'])

stage_mode = peak_stage.merge(mode[['angle','tensile_fraction']], on='angle')
display(stage_mode)
display(bootstrap)


In [ ]:
def save_figure(fig, stem):
    for ext in CONFIG['formats']:
        path = OUT / f'{stem}.{ext}'
        kwargs = dict(bbox_inches='tight', transparent=CONFIG['transparent'])
        if ext.lower() in {'png','jpg','jpeg','tif','tiff'}:
            kwargs['dpi'] = CONFIG['dpi']
        fig.savefig(path, **kwargs)
        print('saved:', path)


## Figure 1 — Mechanical trajectories and cumulative AE energy

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=CONFIG['fig1_size'])
for angle in ANGLES:
    d = data[angle]
    axs[0].plot(d['load_t']/d['tf'], d['load']/d['peak'], lw=CONFIG['line_width'], label=f'{angle}°')
    order = np.argsort(d['ae_t'])
    t = d['ae_t'][order]
    e = d['ae_e'][order]
    cum = np.cumsum(e)
    if len(cum) and cum[-1] > 0:
        axs[1].plot(t/d['tf'], cum/cum[-1], lw=CONFIG['line_width'], label=f'{angle}°')

for ax in axs:
    ax.axvline(1.0, ls='--', lw=1.0)
    ax.set_xlim(left=0)
axs[0].set_xlabel(r'Normalized time, $t/t_f$')
axs[0].set_ylabel('Normalized load')
axs[1].set_xlabel(r'Normalized time, $t/t_f$')
axs[1].set_ylabel('Normalized cumulative AE energy')
if CONFIG['show_titles']:
    axs[0].set_title('Mechanical approach to peak load')
    axs[1].set_title('AE-energy accumulation')
axs[0].legend(ncol=2, frameon=False)
fig.tight_layout()
save_figure(fig, 'Figure_1_mechanics_and_AE')
plt.show()


## Figure 2 — Stage of maximum AE-energy rate

In [ ]:
fig, ax = plt.subplots(figsize=CONFIG['fig2_size'])
ax.bar(peak_stage['angle'].astype(str)+'°', peak_stage['stage_of_peak_energy_rate'])
ax.set_ylim(2.5, 4.5)
ax.set_yticks([3,4], ['Stage III','Stage IV'])
ax.set_xlabel('Bedding angle')
ax.set_ylabel('Stage of maximum AE-energy rate')
if CONFIG['show_titles']:
    ax.set_title('Timing of peak AE-energy release')
fig.tight_layout()
save_figure(fig, 'Figure_2_peak_stage_by_angle')
plt.show()


## Figure 3 — Block-bootstrap uncertainty

In [ ]:
x = np.arange(len(bootstrap))
yerr = np.vstack([
    bootstrap['p_hat'] - bootstrap['ci_low'],
    bootstrap['ci_high'] - bootstrap['p_hat'],
])
fig, ax = plt.subplots(figsize=CONFIG['fig3_size'])
ax.errorbar(x, bootstrap['p_hat'], yerr=yerr, fmt='o', ms=CONFIG['marker_size'], capsize=CONFIG['capsize'])
ax.axhline(0, ls='--', lw=1.0)
ax.set_xticks(x, [f'{a}°' for a in bootstrap['angle']])
ax.set_xlabel('Bedding angle')
ax.set_ylabel(r'Acceleration exponent estimate, $p$')
if CONFIG['show_titles']:
    ax.set_title('Block-bootstrap uncertainty')
fig.tight_layout()
save_figure(fig, 'Figure_3_bootstrap_exponent')
plt.show()


## Figure 4 — Fracture-mode composition versus peak-energy stage

In [ ]:
fig, ax = plt.subplots(figsize=CONFIG['fig4_size'])
ax.scatter(stage_mode['tensile_fraction'], stage_mode['stage_of_peak_energy_rate'], s=CONFIG['marker_size']**2)
for _, r in stage_mode.iterrows():
    ax.annotate(f"{int(r['angle'])}°", (r['tensile_fraction'], r['stage_of_peak_energy_rate']),
                xytext=(5,4), textcoords='offset points')
ax.set_yticks([3,4], ['Stage III','Stage IV'])
ax.set_xlabel('Tensile-event fraction')
ax.set_ylabel('Stage of maximum AE-energy rate')
if CONFIG['show_titles']:
    ax.set_title('Fracture mode and peak-energy timing')
fig.tight_layout()
save_figure(fig, 'Figure_4_mode_vs_peak_stage')
plt.show()


## 4. Optional: download all generated figures as a ZIP

In [ ]:
import shutil
zip_name = '/content/acoustic_anisotropy_figures'
archive = shutil.make_archive(zip_name, 'zip', OUT)
print(archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    pass
